In [1]:
import pandas as pd
import json
import numpy as np
import re
import ast
import unicodedata
import math
import networkx as nx

from pathlib import Path
from collections import Counter, defaultdict

### authorsPerPaperData

In [2]:
# Load the dataset
SEC2A_CSV = Path('../data/processed/outputs/openalex_notebook_outputs/tables/sec2a_authors_per_paper_by_year.csv')
OUT_JS = Path('../site/data/authors/authorsPerPaperData.js')

sec2a = pd.read_csv(SEC2A_CSV)
print('Loaded:', SEC2A_CSV)

ROUND_AVG = 2
ROUND_BAND = 2

Loaded: ../data/processed/outputs/openalex_notebook_outputs/tables/sec2a_authors_per_paper_by_year.csv


In [3]:
# =========================
# Build authorsPerPaperData (shape expected by authors/plots/authorsPerPaper.js)
# min/max band: mean ± std (like your mock)
# =========================
required = ['Year','mean_authors','std_authors','min_authors','max_authors','n_papers']
missing = [c for c in required if c not in sec2a.columns]
if missing:
    raise ValueError('sec2a missing columns: ' + ', '.join(missing))

sec2a['Year'] = pd.to_numeric(sec2a['Year'], errors='coerce')
sec2a = sec2a.dropna(subset=['Year']).copy()
sec2a['Year'] = sec2a['Year'].astype(int)

sec2a['mean_authors'] = pd.to_numeric(sec2a['mean_authors'], errors='coerce')
sec2a['std_authors'] = pd.to_numeric(sec2a['std_authors'], errors='coerce').fillna(0.0)

authorsPerPaperData = []
for _, r in sec2a.sort_values('Year').iterrows():
    year = int(r['Year'])
    avg = float(r['mean_authors']) if pd.notna(r['mean_authors']) else 0.0
    band = float(r['std_authors']) if pd.notna(r['std_authors']) else 0.0
    mn = max(avg - band, 0.0)
    mx = avg + band
    authorsPerPaperData.append({
        'year': year,
        'avg': round(avg, ROUND_AVG),
        'variance': round(band, ROUND_BAND),  # NOTE: this is STD (band), not mathematical variance
        'min': round(mn, ROUND_BAND),
        'max': round(mx, ROUND_BAND),
    })

authorsPerPaperData[:3], authorsPerPaperData[-3:]

# =========================
# Stats (for annotation)
# =========================
years = [d['year'] for d in authorsPerPaperData]
avgs  = [d['avg'] for d in authorsPerPaperData]

overallAvg = float(np.mean(avgs)) if avgs else 0.0

trend = 'flat'
if len(years) >= 2:
    x = np.array(years, dtype=float)
    y = np.array(avgs, dtype=float)
    slope = np.polyfit(x, y, 1)[0]
    if slope > 0.001:
        trend = 'increasing'
    elif slope < -0.001:
        trend = 'decreasing'

growthRate = 0
if len(avgs) >= 2 and avgs[0] > 0:
    growthRate = int(round(((avgs[-1] - avgs[0]) / avgs[0]) * 100))

authorsPerPaperStats = {
    'overallAvg': round(overallAvg, 2),
    'trend': trend,
    'growthRate': growthRate,
}

authorsPerPaperStats
# =========================
# WRITE JS
# =========================
OUT_JS.parent.mkdir(parents=True, exist_ok=True)

lines = [
  '/**',
  ' * data/authors/authorsPerPaperData.js',
  ' * Average number of authors per paper by year with variance band for error area',
  ' * AUTO-GENERATED from OpenAlex outputs',
  ' */',
  '',
  'export const authorsPerPaperData = ' + json.dumps(authorsPerPaperData, ensure_ascii=False, indent=2) + ';',
  '',
  'export const authorsPerPaperStats = ' + json.dumps(authorsPerPaperStats, ensure_ascii=False, indent=2) + ';',
  '',
]

OUT_JS.write_text('\n'.join(lines), encoding='utf-8')
print('✅ Wrote:', OUT_JS.resolve())
print('Years:', years[0], '→', years[-1], '| growthRate:', authorsPerPaperStats['growthRate'], '%')



✅ Wrote: /Users/irynasavchuk/Desktop/DATAVIZ_PROJECT/DV/dv_repo/site/data/authors/authorsPerPaperData.js
Years: 1990 → 2024 | growthRate: 108 %


### uniqueAuthorsData

In [4]:
# INPUT / OUTPUT
# =========================
SRC = Path("../data/processed/outputs/openalex_notebook_outputs/tables/dataset_clean_with_openalex.csv")
OUT = Path("../data/processed/outputs/openalex_notebook_outputs/tables/sec2b_unique_authors_cumulative.csv")

# =========================
# Helpers
# =========================
def norm_spaces(s: str) -> str:
    s = unicodedata.normalize("NFKC", s)
    s = s.strip()
    s = re.sub(r"\s+", " ", s)
    return s

def split_authors(cell) -> list[str]:
    if pd.isna(cell):
        return []
    s = str(cell)
    parts = [p.strip() for p in s.split(";")]
    parts = [norm_spaces(p) for p in parts if p and p.strip()]
    return parts

def author_key(name: str) -> str:
    # chiave per dedup robusta (case-insensitive)
    return norm_spaces(name).casefold()

# =========================
# Load
# =========================
df = pd.read_csv(SRC)

if "Year" not in df.columns or "AuthorNames-Deduped-Clean" not in df.columns:
    raise ValueError("Mancano colonne 'Year' o 'AuthorNames-Deduped-Clean' nel dataset.")

df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype("Int64")
df = df.dropna(subset=["Year"]).copy()
df["Year"] = df["Year"].astype(int)

# =========================
# Explode year-author pairs
# =========================
rows = []
for _, r in df.iterrows():
    y = int(r["Year"])
    authors = split_authors(r["AuthorNames-Deduped-Clean"])
    for a in authors:
        rows.append((y, author_key(a), a))  # (year, key, display)

pairs = pd.DataFrame(rows, columns=["Year", "author_key", "author_display"])
pairs = pairs.drop_duplicates(subset=["Year", "author_key"])  # unico per anno

# =========================
# Per-year unique sets
# =========================
year_to_set = (pairs.groupby("Year")["author_key"]
               .apply(lambda s: set(s.tolist()))
               .to_dict())

years = sorted(year_to_set.keys())

seen = set()
out_rows = []

for y in years:
    current = year_to_set[y]
    unique_year = len(current)
    new_authors = len(current - seen)
    seen |= current
    cum = len(seen)

    out_rows.append({
        "Year": y,
        "unique_authors_year": unique_year,
        "unique_authors_new": new_authors,
        "unique_authors_cum": cum
    })

out_df = pd.DataFrame(out_rows)

# =========================
# Save
# =========================
OUT.parent.mkdir(parents=True, exist_ok=True)
out_df.to_csv(OUT, index=False)

print("✅ Wrote:", OUT.resolve())
print("Total unique authors (overall):", out_df["unique_authors_cum"].iloc[-1])
print("Sanity check (should match):", pairs["author_key"].nunique())


✅ Wrote: /Users/irynasavchuk/Desktop/DATAVIZ_PROJECT/DV/dv_repo/data/processed/outputs/openalex_notebook_outputs/tables/sec2b_unique_authors_cumulative.csv
Total unique authors (overall): 6833
Sanity check (should match): 6833


In [5]:
SEC2B_CSV = Path("../data/processed/outputs/openalex_notebook_outputs/tables/sec2b_unique_authors_cumulative.csv")
OUT_JS = Path("../site/data/authors/uniqueAuthorsData.js")

In [6]:
# =========================
# LOAD
# =========================
df = pd.read_csv(SEC2B_CSV)

# Normalize column names (tolerant to small variations)
df = df.rename(columns={
    "year": "Year",
    "unique_authors": "unique_authors_year",
    "unique_authors_cumulative": "unique_authors_cum",
})

required = {"Year", "unique_authors_year", "unique_authors_cum"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing columns in sec2b file: {sorted(missing)}. Found: {list(df.columns)}")

# Coerce types
df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype("Int64")
df["unique_authors_year"] = pd.to_numeric(df["unique_authors_year"], errors="coerce")
df["unique_authors_cum"] = pd.to_numeric(df["unique_authors_cum"], errors="coerce")

df = df.dropna(subset=["Year", "unique_authors_year", "unique_authors_cum"]).copy()
df = df.sort_values("Year")

df.head()

# =========================
# BUILD DATA (format expected by the plot)
# =========================
uniqueAuthorsData = []
for _, r in df.iterrows():
    uniqueAuthorsData.append({
        "year": int(r["Year"]),
        "cumulative": int(round(float(r["unique_authors_cum"]))),
        "newAuthors": int(round(float(r["unique_authors_year"]))),
    })

# =========================
# STATS
# =========================
total = uniqueAuthorsData[-1]["cumulative"] if uniqueAuthorsData else 0
avg_new = int(round(df["unique_authors_year"].mean())) if len(df) else 0
peak_idx = int(df["unique_authors_year"].idxmax()) if len(df) else None
peak_year = int(df.loc[peak_idx, "Year"]) if peak_idx is not None else None
peak_new = int(round(float(df.loc[peak_idx, "unique_authors_year"]))) if peak_idx is not None else 0

uniqueAuthorsStats = {
    "total": int(total),
    "avgNewPerYear": int(avg_new),
    "peakYear": peak_year,
    "peakNewAuthors": int(peak_new),
}

uniqueAuthorsStats

# =========================
# WRITE JS
# =========================
OUT_JS.parent.mkdir(parents=True, exist_ok=True)

lines = [
    "/**",
    " * data/authors/uniqueAuthorsData.js",
    " * Cumulative count of unique authors by year",
    " * AUTO-GENERATED from sec2b_unique_authors_cumulative.csv",
    " */",
    "",
    "export const uniqueAuthorsData = " + json.dumps(uniqueAuthorsData, ensure_ascii=False, indent=2) + ";",
    "",
    "export const uniqueAuthorsStats = " + json.dumps(uniqueAuthorsStats, ensure_ascii=False, indent=2) + ";",
    "",
]

OUT_JS.write_text("\n".join(lines), encoding="utf-8")
print("✅ Wrote:", OUT_JS.resolve())
print("Years:", uniqueAuthorsData[0]["year"], "→", uniqueAuthorsData[-1]["year"]) if uniqueAuthorsData else None
print("Total unique authors:", uniqueAuthorsStats["total"])
print("Avg new authors/year:", uniqueAuthorsStats["avgNewPerYear"])
print("Peak year:", uniqueAuthorsStats["peakYear"], "(new:", uniqueAuthorsStats["peakNewAuthors"], ")")


✅ Wrote: /Users/irynasavchuk/Desktop/DATAVIZ_PROJECT/DV/dv_repo/site/data/authors/uniqueAuthorsData.js
Years: 1990 → 2024
Total unique authors: 6833
Avg new authors/year: 346
Peak year: 2020 (new: 648 )


### authorMetricsData

In [7]:
# =========================
# INPUT / OUTPUT
# =========================
SRC = Path("../data/processed/outputs/openalex_notebook_outputs/tables/sec2d_author_stats.csv")
OUT_JS = Path("../site/data/authors/authorMetricsData.js")

TOP_N = 50  # cambia a 30 / 80 / 200 ecc.

# =========================
# Load + clean
# =========================
df = pd.read_csv(SRC)

needed = {"author_id", "display_name", "n_papers", "total_citations", "n_awards"}
missing = needed - set(df.columns)
if missing:
    raise ValueError(f"Mancano colonne in sec2d_author_stats.csv: {missing}")

df["n_papers"] = pd.to_numeric(df["n_papers"], errors="coerce").fillna(0).astype(int)
df["total_citations"] = pd.to_numeric(df["total_citations"], errors="coerce").fillna(0)
df["n_awards"] = pd.to_numeric(df["n_awards"], errors="coerce").fillna(0).astype(int)

df["display_name"] = df["display_name"].fillna("").astype(str).str.strip()
df = df[df["display_name"] != ""].copy()

# =========================
# Selezione TOP_N bilanciata
# (unione top per papers + top per citations)
# =========================
k = max(TOP_N, 1)

top_p = df.nlargest(k, "n_papers")
top_c = df.nlargest(k, "total_citations")
sel = pd.concat([top_p, top_c], ignore_index=True).drop_duplicates(subset=["author_id"])

# ranking combinato (più robusto di una somma grezza)
sel["rank_papers"] = sel["n_papers"].rank(method="min", ascending=False)
sel["rank_citations"] = sel["total_citations"].rank(method="min", ascending=False)
sel["rank_score"] = sel["rank_papers"] + sel["rank_citations"]

sel = sel.sort_values(["rank_score", "n_papers", "total_citations"], ascending=[True, False, False]).head(k).copy()

# =========================
# Categorie (soglie adattive)
# =========================
# soglie basate sui selezionati (così il grafico ha sempre varietà)
papers_prolific = int(max(1, math.floor(sel["n_papers"].quantile(0.85))))
cit_high = float(sel["total_citations"].quantile(0.85))
papers_steady = int(max(1, math.floor(sel["n_papers"].quantile(0.50))))

def category(row) -> str:
    if row["total_citations"] >= cit_high and row["total_citations"] > 0:
        return "highly-cited"
    if row["n_papers"] >= papers_prolific and row["n_papers"] > 0:
        return "prolific"
    if row["n_papers"] >= papers_steady and row["n_papers"] > 0:
        return "steady"
    return "emerging"

sel["category"] = sel.apply(category, axis=1)

# =========================
# Build JS data
# =========================
data = []
for _, r in sel.iterrows():
    data.append({
        "name": r["display_name"],
        "papers": int(r["n_papers"]),
        "citations": int(round(float(r["total_citations"]))),
        "awards": int(r["n_awards"]),
        "category": r["category"],
    })

# ordina (opzionale): più papers, poi più citations
data = sorted(data, key=lambda x: (-x["papers"], -x["citations"], x["name"]))

# =========================
# Stats
# =========================
total_authors = len(data)
avg_papers = int(round(sum(d["papers"] for d in data) / total_authors)) if total_authors else 0
avg_cit = int(round(sum(d["citations"] for d in data) / total_authors)) if total_authors else 0
avg_awards = round(sum(d["awards"] for d in data) / total_authors, 2) if total_authors else 0

max_awards_obs = max([d["awards"] for d in data], default=0)
max_awards_safe = max(1, int(max_awards_obs))  # IMPORTANT per la sizeScale nel chart

authorMetricsStats = {
    "totalAuthors": int(total_authors),
    "avgPapers": int(avg_papers),
    "avgCitations": int(avg_cit),
    "avgAwards": float(avg_awards),
    "maxAwards": int(max_awards_safe),
    "categories": {
        "prolific": f">= {papers_prolific} papers",
        "highly-cited": f">= {int(round(cit_high))} citations",
        "steady": f"{papers_steady}–{max(papers_prolific - 1, papers_steady)} papers",
        "emerging": f"< {papers_steady} papers"
    }
}

# =========================
# WRITE JS (newlines vere)
# =========================
OUT_JS.parent.mkdir(parents=True, exist_ok=True)

lines = [
    "/**",
    " * data/authors/authorMetricsData.js",
    " * Author metrics for bubble scatter chart",
    " * X-axis: number of papers, Y-axis: citations, Bubble size: awards",
    " * AUTO-GENERATED from sec2d_author_stats.csv",
    " */",
    "",
    "export const authorMetricsData = " + json.dumps(data, ensure_ascii=False, indent=2) + ";",
    "",
    "export const authorMetricsStats = " + json.dumps(authorMetricsStats, ensure_ascii=False, indent=2) + ";",
    "",
]

OUT_JS.write_text("\n".join(lines), encoding="utf-8")
print("✅ Wrote:", OUT_JS.resolve())
print("Authors:", total_authors, "| maxAwards(obs):", max_awards_obs, "| maxAwards(used):", max_awards_safe)


✅ Wrote: /Users/irynasavchuk/Desktop/DATAVIZ_PROJECT/DV/dv_repo/site/data/authors/authorMetricsData.js
Authors: 50 | maxAwards(obs): 10 | maxAwards(used): 10


### collaborationNetwork

In [8]:
# ----------------------------
# CONFIG
# ----------------------------
# Adjust this to your repo root if needed:
REPO_ROOT = Path("..")  # e.g., Path("/path/to/dv_repo_parent")

TABLES_DIR = REPO_ROOT/ "data" / "processed" / "outputs" / "openalex_notebook_outputs" / "tables"

AUTHORS_PATH        = TABLES_DIR / "authors.csv"
AUTHORSHIPS_PATH    = TABLES_DIR / "authorships.csv"       # optional but recommended
COAUTHOR_EDGES_PATH = TABLES_DIR / "coauthor_edges.csv"

# Optional (not required to build the network; kept here for future enrichment)
DATASET_PATH = TABLES_DIR / "dataset_clean_with_openalex.csv"

# Visual clarity: keep only the top N most collaborative authors
TOP_N_AUTHORS = 100

# Keep only the strongest edges (optional). Set to 1 to keep all.
MIN_EDGE_WEIGHT = 1

# Output directory
OUT_DIR = REPO_ROOT / "site" / "data" / "authors" 
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Tables dir:", TABLES_DIR.resolve())
print("Output dir:", OUT_DIR.resolve())

Tables dir: /Users/irynasavchuk/Desktop/DATAVIZ_PROJECT/DV/dv_repo/data/processed/outputs/openalex_notebook_outputs/tables
Output dir: /Users/irynasavchuk/Desktop/DATAVIZ_PROJECT/DV/dv_repo/site/data/authors


In [9]:
def _assert_exists(path: Path):
    if not path.exists():
        raise FileNotFoundError(
            f"Missing file: {path}\n"
            f"Double-check REPO_ROOT and TABLES_DIR in the config cell above."
        )

_assert_exists(AUTHORS_PATH)
_assert_exists(COAUTHOR_EDGES_PATH)

authors = pd.read_csv(AUTHORS_PATH)
coedges = pd.read_csv(COAUTHOR_EDGES_PATH)

# authorships is optional
authorships = None
if AUTHORSHIPS_PATH.exists():
    authorships = pd.read_csv(AUTHORSHIPS_PATH)
else:
    print("Note: authorships.csv not found — 'papers' and group labels may be less informative.")

print("authors:", authors.shape)
print("coauthor_edges:", coedges.shape)
if authorships is not None:
    print("authorships:", authorships.shape)

authors.head()

authors: (7200, 5)
coauthor_edges: (23747, 4)
authorships: (14189, 11)


,author_id,openalex_author_id_url,display_name,orcid,source
0,A5023631793,https://openalex.org/A5023631793,Rainer Splechtna,NaN,openalex
1,A5030567814,https://openalex.org/A5030567814,Majid Behravan,https://orcid.org/0000-0001-6525-6646,openalex
2,A5007628057,https://openalex.org/A5007628057,Mario Jelović,NaN,openalex
3,A5056069234,https://openalex.org/A5056069234,Denis Gračanin,https://orcid.org/0000-0001-6831-2818,openalex
4,A5000755776,https://openalex.org/A5000755776,Helwig Hauser,https://orcid.org/0000-0003-0395-3192,openalex


In [10]:
def parse_list_field(x):
    # Parse list-like fields stored as strings in CSV (JSON-ish or Python-literal-ish).
    # Returns a Python list.
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []
    if isinstance(x, list):
        return x
    s = str(x).strip()
    if s == "" or s.lower() == "nan":
        return []

    # Try JSON as-is
    if s.startswith("[") and s.endswith("]"):
        try:
            return json.loads(s)
        except Exception:
            pass

        # Handle doubled quotes from CSV (e.g., ["","x"",""])
        try:
            s2 = s.replace('""', '"')
            return json.loads(s2)
        except Exception:
            pass

        # Try Python literal eval
        try:
            return ast.literal_eval(s)
        except Exception:
            pass

        # Last resort: split
        inner = s[1:-1].strip()
        if not inner:
            return []
        parts = [p.strip().strip('"').strip("'") for p in inner.split(",")]
        return [p for p in parts if p]

    # Not a list string
    return [s]

# Demonstrate parsing on a couple fields if present
if "doi_list" in coedges.columns:
    sample = coedges["doi_list"].dropna().head(1).tolist()
    if sample:
        print("doi_list raw:", sample[0])
        print("doi_list parsed:", parse_list_field(sample[0])[:3], "...")

doi_list raw: ["10.1109/tvcg.2009.178", "10.1109/tvcg.2012.240", "10.1109/tvcg.2013.142", "10.1109/tvcg.2014.2346312", "10.1109/tvcg.2014.2346371", "10.1109/tvcg.2015.2467441", "10.1109/tvcg.2017.2744238", "10.1109/tvcg.2017.2744278", "10.1109/tvcg.2017.2745941", "10.1109/tvcg.2018.2864847", "10.1109/tvcg.2019.2934547", "10.1109/tvcg.2020.3030427", "10.1109/tvcg.2020.3030454", "10.1109/tvcg.2021.3114845", "10.1109/tvcg.2022.3209353", "10.1109/tvcg.2022.3209479", "10.1109/tvcg.2022.3209497", "10.1109/tvcg.2023.3326568", "10.1109/tvcg.2023.3326579", "10.1109/tvcg.2023.3327161", "10.1109/tvcg.2023.3327193", "10.1109/tvcg.2023.3327388", "10.1109/tvcg.2024.3456406"]
doi_list parsed: ['10.1109/tvcg.2009.178', '10.1109/tvcg.2012.240', '10.1109/tvcg.2013.142'] ...


In [11]:
# ----------------------------
# Author name map
# ----------------------------
# Prefer authors.display_name; fall back to the name found in authorships if needed.
author_id_to_name = dict(
    zip(
        authors["author_id"],
        authors["display_name"] if "display_name" in authors.columns else authors["author_id"]
    )
)

if authorships is not None and "author_id" in authorships.columns and "author_display_name" in authorships.columns:
    # fill missing display names
    for aid, nm in (
        authorships[["author_id", "author_display_name"]]
        .dropna()
        .drop_duplicates()
        .itertuples(index=False)
    ):
        author_id_to_name.setdefault(aid, nm)

# ----------------------------
# Compute collaboration strength per author (weighted degree from coauthor_edges)
# ----------------------------
coedges["weight"] = pd.to_numeric(coedges["weight"], errors="coerce").fillna(0).astype(float)

coedges_f = coedges[coedges["weight"] >= MIN_EDGE_WEIGHT].copy()

collab_strength = (
    pd.concat([
        coedges_f[["author_id_1", "weight"]].rename(columns={"author_id_1": "author_id"}),
        coedges_f[["author_id_2", "weight"]].rename(columns={"author_id_2": "author_id"}),
    ])
    .groupby("author_id")["weight"]
    .sum()
    .sort_values(ascending=False)
)

# ----------------------------
# Papers per author (optional, from authorships)
# ----------------------------
papers_per_author = None
if authorships is not None and "work_id" in authorships.columns:
    papers_per_author = (
        authorships.dropna(subset=["author_id", "work_id"])
        .groupby("author_id")["work_id"]
        .nunique()
        .sort_values(ascending=False)
    )

metrics = pd.DataFrame({
    "author_id": collab_strength.index,
    "collaboration_strength": collab_strength.values,
})

if papers_per_author is not None:
    metrics["papers"] = metrics["author_id"].map(papers_per_author).fillna(0).astype(int)
else:
    metrics["papers"] = 0

metrics["display_name"] = metrics["author_id"].map(author_id_to_name).fillna(metrics["author_id"])

metrics.head(10)

,author_id,collaboration_strength,papers,display_name
0,A5091466289,353.0,70,Huamin Qu
1,A5043151044,307.0,58,Hanspeter Pfister
2,A5073986937,284.0,50,Yingcai Wu
3,A5100344494,180.0,27,Wei Chen
4,A5009460413,177.0,41,Valerio Pascucci
5,A5073919282,171.0,44,Daniel A. Keim
6,A5109928029,163.0,39,M. Eduard Gröller
7,A5037161857,159.0,63,Kwan‐liu Ma
8,A5018584803,156.0,35,David S. Ebert
9,A5038629539,152.0,45,Thomas Ertl


In [12]:
# ----------------------------
# Select top authors + build induced subgraph edges
# ----------------------------
top_author_ids = metrics.sort_values("collaboration_strength", ascending=False).head(TOP_N_AUTHORS)["author_id"].tolist()
top_set = set(top_author_ids)

sub_edges = coedges_f[
    coedges_f["author_id_1"].isin(top_set) & coedges_f["author_id_2"].isin(top_set)
].copy()

print("Top authors:", len(top_author_ids))
print("Edges in induced subgraph:", sub_edges.shape[0])

# ----------------------------
# Build NetworkX graph for community detection
# ----------------------------
G = nx.Graph()
for aid in top_author_ids:
    G.add_node(aid)

for a1, a2, w in sub_edges[["author_id_1", "author_id_2", "weight"]].itertuples(index=False):
    if a1 == a2:
        continue
    if G.has_edge(a1, a2):
        G[a1][a2]["weight"] = max(G[a1][a2]["weight"], float(w))
    else:
        G.add_edge(a1, a2, weight=float(w))

print("Graph nodes:", G.number_of_nodes(), "edges:", G.number_of_edges())

Top authors: 100
Edges in induced subgraph: 389
Graph nodes: 100 edges: 389


In [13]:
# ----------------------------
# Community detection → up to 3 groups
# ----------------------------
from networkx.algorithms.community import greedy_modularity_communities

communities = list(greedy_modularity_communities(G, weight="weight"))
communities = sorted(communities, key=len, reverse=True)

print("Detected communities:", len(communities), "sizes:", [len(c) for c in communities[:10]])

# Keep 3 largest; merge the rest into group 3
communities_3 = communities[:3]
if len(communities) > 3:
    merged = set().union(*communities[3:])
    communities_3 = communities[:2] + [set(communities[2]).union(merged)]

node_group = {}
for gi, comm in enumerate(communities_3, start=1):
    for aid in comm:
        node_group[aid] = gi

for aid in G.nodes():
    node_group.setdefault(aid, 3)

from collections import Counter
Counter(node_group.values())

Detected communities: 10 sizes: [20, 14, 13, 13, 11, 10, 9, 6, 3, 1]


Counter({3: 66, 1: 20, 2: 14})

In [14]:
# ----------------------------
# Build human-ish group labels (top institutions/countries per group)
# ----------------------------
def most_common_flat(list_of_lists, n=2):
    c = Counter()
    for lst in list_of_lists:
        for x in lst:
            if x:
                c[x] += 1
    return [k for k, _ in c.most_common(n)]

group_label = {1: "Community 1", 2: "Community 2", 3: "Community 3"}

if authorships is not None:
    inst_names = None
    country_codes = None

    if "institutions_names" in authorships.columns:
        inst_names = authorships[["author_id", "institutions_names"]].dropna()
        inst_names["institutions_names"] = inst_names["institutions_names"].map(parse_list_field)

    if "institutions_country_codes" in authorships.columns:
        country_codes = authorships[["author_id", "institutions_country_codes"]].dropna()
        country_codes["institutions_country_codes"] = country_codes["institutions_country_codes"].map(parse_list_field)

    author_to_insts = defaultdict(list)
    author_to_countries = defaultdict(list)

    if inst_names is not None:
        for aid, insts in inst_names.itertuples(index=False):
            author_to_insts[aid].append(insts)

    if country_codes is not None:
        for aid, cc in country_codes.itertuples(index=False):
            author_to_countries[aid].append(cc)

    for gi in [1, 2, 3]:
        members = [aid for aid in top_author_ids if node_group.get(aid) == gi]

        insts_top = most_common_flat([x for aid in members for x in author_to_insts.get(aid, [])], n=2)
        cc_top = most_common_flat([x for aid in members for x in author_to_countries.get(aid, [])], n=2)

        bits = []
        if insts_top:
            bits.append("top inst: " + ", ".join(insts_top))
        if cc_top:
            bits.append("top country: " + ", ".join(cc_top))

        if bits:
            group_label[gi] = f"Community {gi} (" + " • ".join(bits) + ")"

group_label

{1: 'Community 1 (top inst: Hong Kong University of Science and Technology, Zhejiang University • top country: CN, HK)',
 2: 'Community 2 (top inst: TU Wien, University of Bergen • top country: AT, DE)',
 3: 'Community 3 (top inst: University of Stuttgart, University of Utah • top country: US, DE)'}

In [15]:
# ----------------------------
# Prepare D3-friendly node/link objects
# ----------------------------
def make_unique_labels(labels, fallback_ids):
    # Ensure labels are unique; if duplicates exist, append a short suffix from author_id.
    seen = Counter(labels)
    out = []
    used = set()
    for lab, aid in zip(labels, fallback_ids):
        if seen[lab] == 1 and lab not in used:
            out.append(lab)
            used.add(lab)
        else:
            suffix = aid[-6:]
            cand = f"{lab} ({suffix})"
            k = 2
            while cand in used:
                cand = f"{lab} ({suffix}-{k})"
                k += 1
            out.append(cand)
            used.add(cand)
    return out

node_df = metrics[metrics["author_id"].isin(top_set)].copy()
node_df["group"] = node_df["author_id"].map(node_group).fillna(3).astype(int)

node_df = node_df.sort_values("collaboration_strength", ascending=False)
node_df["id"] = make_unique_labels(node_df["display_name"].tolist(), node_df["author_id"].tolist())

author_id_to_d3id = dict(zip(node_df["author_id"], node_df["id"]))

nodes = []
for row in node_df.itertuples(index=False):
    nodes.append({
        "id": row.id,
        "group": int(row.group),
        "papers": int(row.papers),
        "collaborations": int(round(row.collaboration_strength)),
        "author_id": row.author_id,  # optional metadata
    })

links = []
for a1, a2, w in sub_edges[["author_id_1", "author_id_2", "weight"]].itertuples(index=False):
    if a1 not in author_id_to_d3id or a2 not in author_id_to_d3id:
        continue
    links.append({
        "source": author_id_to_d3id[a1],
        "target": author_id_to_d3id[a2],
        "value": float(w),
    })

avg_link_weight = float(np.mean([l["value"] for l in links])) if links else 0.0
max_collab = int(max(n["collaborations"] for n in nodes)) if nodes else 0

network_stats = {
    "totalNodes": len(nodes),
    "totalLinks": len(links),
    "avgCollaborations": round(avg_link_weight, 2),
    "maxCollaborations": max_collab,
    "groups": {str(k): v for k, v in group_label.items()},
}

collaboration_network_data = {"nodes": nodes, "links": links}

print("network_stats:", network_stats)
print("Example node:", nodes[0] if nodes else None)
print("Example link:", links[0] if links else None)

network_stats: {'totalNodes': 100, 'totalLinks': 389, 'avgCollaborations': 2.56, 'maxCollaborations': 353, 'groups': {'1': 'Community 1 (top inst: Hong Kong University of Science and Technology, Zhejiang University • top country: CN, HK)', '2': 'Community 2 (top inst: TU Wien, University of Bergen • top country: AT, DE)', '3': 'Community 3 (top inst: University of Stuttgart, University of Utah • top country: US, DE)'}}
Example node: {'id': 'Huamin Qu', 'group': 1, 'papers': 70, 'collaborations': 353, 'author_id': 'A5091466289'}
Example link: {'source': 'Johanna Beyer', 'target': 'Hanspeter Pfister', 'value': 23.0}


In [16]:
# ----------------------------
# Write outputs
# ----------------------------
json_path = OUT_DIR / "collaborationNetworkData.json"
stats_path = OUT_DIR / "networkStats.json"
js_path = OUT_DIR / "collaborationNetworkData.js"

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(collaboration_network_data, f, ensure_ascii=False, indent=2)

with open(stats_path, "w", encoding="utf-8") as f:
    json.dump(network_stats, f, ensure_ascii=False, indent=2)

js = (
    "/* Auto-generated by build_collaboration_network.ipynb */\n\n"
    "export const collaborationNetworkData = "
    + json.dumps(collaboration_network_data, ensure_ascii=False, indent=2)
    + ";\n\n"
    "export const networkStats = "
    + json.dumps(network_stats, ensure_ascii=False, indent=2)
    + ";\n"
)

with open(js_path, "w", encoding="utf-8") as f:
    f.write(js)

print("Wrote:")
print(" -", json_path)
print(" -", stats_path)
print(" -", js_path)

Wrote:
 - ../site/data/authors/collaborationNetworkData.json
 - ../site/data/authors/networkStats.json
 - ../site/data/authors/collaborationNetworkData.js
